# Neo4j UC Federation Prototype: Validation Test Suite

## Overview

This notebook validates Neo4j connectivity through Unity Catalog's generic JDBC path (`TYPE JDBC`). It provides a systematic test progression from basic network connectivity through the full UC JDBC connection, documenting what works and where a first-class `TYPE NEO4J` connector would streamline the experience.

**Key finding**: The prototype is fully functional. During development we worked with Databricks engineering to resolve a SafeSpark sandbox memory configuration (Section 6) by adding three Spark configuration properties. The test progression below was the methodology used to isolate that issue from other potential failure points.

All tests ran on a live Databricks cluster connected to Neo4j Aura.

---

## Configuration

**Prerequisites**: Run `setup.sh` to configure Databricks secrets before running this notebook.

The setup script reads credentials from `.env` and stores them in the `neo4j-uc-creds` secret scope:
- `host` - Neo4j host
- `user` - Neo4j username
- `password` - Neo4j password
- `connection_name` - Unity Catalog connection name
- `jdbc_jar_path` - Path to Neo4j Unity Catalog Connector JAR in UC Volume
- `database` - Neo4j database (optional, defaults to "neo4j")

In [ ]:
# =============================================================================
# CONFIGURATION - Loaded from Databricks Secrets
# =============================================================================
# Secrets are configured using setup.sh which creates scope "neo4j-uc-creds"
# with secrets: host, user, password, connection_name, jdbc_jar_path, database

SCOPE_NAME = "neo4j-uc-creds"

# Aura Connection Details (from secrets)
NEO4J_HOST = dbutils.secrets.get(SCOPE_NAME, "host")
NEO4J_USER = dbutils.secrets.get(SCOPE_NAME, "user")
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")

# Database defaults to "neo4j" if not set
try:
    NEO4J_DATABASE = dbutils.secrets.get(SCOPE_NAME, "database")
except:
    NEO4J_DATABASE = "neo4j"

# Unity Catalog Resources (from secrets)
JDBC_JAR_PATH = dbutils.secrets.get(SCOPE_NAME, "jdbc_jar_path")
UC_CONNECTION_NAME = dbutils.secrets.get(SCOPE_NAME, "connection_name")

# Single JAR dependency for CREATE CONNECTION
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

# Derived URLs (no need to edit)
NEO4J_BOLT_URI = f"neo4j+s://{NEO4J_HOST}"
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/{NEO4J_DATABASE}"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"

print("Configuration loaded from Databricks Secrets:")
print(f"  Secret Scope: {SCOPE_NAME}")
print(f"  Neo4j Host: {NEO4J_HOST}")
print(f"  Bolt URI: {NEO4J_BOLT_URI}")
print(f"  JDBC URL: {NEO4J_JDBC_URL}")
print(f"  Connection Name: {UC_CONNECTION_NAME}")
print(f"  JDBC JAR Path: {JDBC_JAR_PATH}")
print(f"  Java Dependencies: {JAVA_DEPENDENCIES}")

---

## Section 1: Environment Information

Capture cluster and runtime details for reproducibility.

In [ ]:
# Collect environment information
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

# Spark version
print(f"\nSpark Version: {spark.version}")

# Databricks Runtime
try:
    dbr_version = spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
    print(f"Databricks Runtime: {dbr_version}")
except:
    print("Databricks Runtime: Unable to determine")

# Python version
import sys
print(f"Python Version: {sys.version}")

# Check neo4j package
try:
    import neo4j
    print(f"Neo4j Python Driver: {neo4j.__version__}")
except ImportError:
    print("Neo4j Python Driver: NOT INSTALLED")

# Check JAR file exists
print(f"\nJDBC JAR Path: {JDBC_JAR_PATH}")
try:
    files = dbutils.fs.ls(JDBC_JAR_PATH.rsplit('/', 1)[0])
    file_names = [f.name for f in files]
    jdbc_jar_found = JDBC_JAR_PATH.split('/')[-1] in file_names
    print(f"JDBC JAR File Exists: {jdbc_jar_found}")
except Exception as e:
    print(f"JAR File Check Error: {e}")

---

## Section 2: Network Connectivity Test (TCP Layer)

**Expected Result**: PASS - Proves network path is open.

In [ ]:
# TCP connectivity test using netcat
import subprocess
import time

print("=" * 60)
print("TEST: Network Connectivity (TCP)")
print("=" * 60)
print(f"\nTarget: {NEO4J_HOST}:7687 (Bolt protocol port)")
print("Testing: Can Databricks reach Neo4j at the network level?")

try:
    start_time = time.time()
    result = subprocess.run(
        ['nc', '-zv', NEO4J_HOST, '7687'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10
    )
    elapsed = (time.time() - start_time) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print("\n" + "=" * 60)
        print(">>> CONNECTIVITY VERIFIED <<<")
        print("=" * 60)
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        print(f"\nConnection Details:")
        print(f"  - Host: {NEO4J_HOST}")
        print(f"  - Port: 7687 (Bolt)")
        print(f"  - TCP Latency: {elapsed:.1f}ms")
        if output:
            print(f"  - Raw Output: {output}")
        print("\n" + "-" * 60)
        print("RESULT: Network path to Neo4j is OPEN")
        print("        Firewall rules allow Bolt protocol traffic")
        print("-" * 60)
        print("\nStatus: PASS")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_HOST}:7687 - check firewall rules")
        print(f"Details: {output}")
        print("\nStatus: FAIL")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")
    print("\nStatus: FAIL")

---

## Section 3: Neo4j Python Driver Test

**Expected Result**: PASS - Proves credentials work and Neo4j is accessible.

In [ ]:
# Test Neo4j Python driver connectivity
import time

print("=" * 60)
print("TEST: Neo4j Python Driver")
print("=" * 60)
print(f"\nTarget: {NEO4J_BOLT_URI}")
print("Testing: Can we authenticate and execute queries via Bolt protocol?")

from neo4j import GraphDatabase

try:
    start_time = time.time()
    driver = GraphDatabase.driver(NEO4J_BOLT_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    
    # Verify connectivity
    driver.verify_connectivity()
    connect_time = (time.time() - start_time) * 1000
    
    print("\n" + "=" * 60)
    print(">>> AUTHENTICATION SUCCESSFUL <<<")
    print("=" * 60)
    print(f"\n[PASS] Driver connected and authenticated in {connect_time:.1f}ms")
    
    # Test simple query
    with driver.session() as session:
        query_start = time.time()
        result = session.run("RETURN 1 AS test")
        record = result.single()
        query_time = (time.time() - query_start) * 1000
        print(f"[PASS] Query executed: RETURN 1 = {record['test']} ({query_time:.1f}ms)")
        
        # Get Neo4j version
        result = session.run("CALL dbms.components() YIELD name, versions RETURN name, versions")
        neo4j_info = []
        for record in result:
            neo4j_info.append(f"{record['name']} {record['versions']}")
    
    total_time = (time.time() - start_time) * 1000
    driver.close()
    
    print(f"\nConnection Details:")
    print(f"  - URI: {NEO4J_BOLT_URI}")
    print(f"  - User: {NEO4J_USER}")
    print(f"  - Database: {NEO4J_DATABASE}")
    print(f"  - Neo4j Server: {', '.join(neo4j_info)}")
    print(f"  - Connection Time: {connect_time:.1f}ms")
    print(f"  - Total Test Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: Neo4j Python Driver connection WORKING")
    print("        Credentials valid, Bolt protocol functional")
    print("-" * 60)
    print("\nStatus: PASS")
    
except Exception as e:
    print(f"\n[FAIL] Connection failed: {e}")
    print("\nStatus: FAIL")

---

## Section 4: Neo4j Spark Connector (Working Baseline)

**Expected Result**: PASS - This is our working baseline that proves Spark can communicate with Neo4j.

In [ ]:
# Test Neo4j Spark Connector (known working method)
import time

print("=" * 60)
print("TEST: Neo4j Spark Connector (org.neo4j.spark.DataSource)")
print("=" * 60)
print(f"\nTarget: {NEO4J_BOLT_URI}")
print("Testing: Can Spark connect to Neo4j using the native Spark Connector?")
print("Method: org.neo4j.spark.DataSource (uses Bolt protocol internally)")

try:
    start_time = time.time()
    df = spark.read.format("org.neo4j.spark.DataSource") \
        .option("url", NEO4J_BOLT_URI) \
        .option("authentication.type", "basic") \
        .option("authentication.basic.username", NEO4J_USER) \
        .option("authentication.basic.password", NEO4J_PASSWORD) \
        .option("query", "RETURN 'Spark Connector Works!' AS message, 1 AS value") \
        .load()
    
    # Force execution and collect results
    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    row_count = len(results)
    
    print("\n" + "=" * 60)
    print(">>> SPARK CONNECTOR WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] Spark Connector query executed successfully in {total_time:.1f}ms")
    
    print(f"\nQuery Results:")
    df.show(truncate=False)
    
    print(f"Connection Details:")
    print(f"  - Connector: org.neo4j.spark.DataSource")
    print(f"  - URL: {NEO4J_BOLT_URI}")
    print(f"  - Auth Type: basic")
    print(f"  - Rows Returned: {row_count}")
    print(f"  - Execution Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: Neo4j Spark Connector WORKING")
    print("        Spark can communicate with Neo4j via Bolt protocol")
    print("        This is the recommended approach for Spark-Neo4j integration")
    print("-" * 60)
    print("\nStatus: PASS")
    
except Exception as e:
    print(f"\n[FAIL] Spark Connector failed: {e}")
    print("\nStatus: FAIL")

---

## Section 5: Direct JDBC Tests (Without SafeSpark Sandbox)

These tests use the Neo4j JDBC driver directly with Spark, **without** Unity Catalog's SafeSpark wrapper.

**Note**: Requires the JDBC JAR to be installed as a cluster library (not just in a UC Volume).

**Limitation Discovered**: Spark's JDBC driver wraps `query` option queries in a subquery for schema inference:
```sql
SELECT * FROM (your_query) SPARK_GEN_SUBQ_N WHERE 1=0
```
This breaks native Cypher even with `FORCE_CYPHER` hint (hint is inside subquery, outer wrapper is still SQL).

**Schema Inference Issue**: When using `dbtable` option, Spark's schema inference returns `NullType()` for all columns from Neo4j JDBC. This causes `No column has been read prior to this call` error when reading data. **Fix**: Use `customSchema` option to explicitly specify column types.

**Workarounds**:
1. Use `dbtable` option with `customSchema` (required to avoid NullType inference)
2. Use `query` option with `customSchema` for SQL queries
3. Use Neo4j Spark Connector instead of JDBC (Section 4 - works without customSchema)

In [ ]:
# Direct JDBC - Using dbtable (reads Neo4j label as table, no subquery wrapping)
import time

print("=" * 60)
print("TEST: Direct JDBC - dbtable option (reads label as table)")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Spark read Neo4j nodes via JDBC using dbtable option?")
print("Method: spark.read.format('jdbc') with Neo4j JDBC Driver")

# Use dbtable to read a Neo4j label directly (no subquery wrapper)
# Replace 'Aircraft' with any label that exists in your Neo4j database
TEST_LABEL = "Aircraft"  # Change this to a label in your database

# IMPORTANT: customSchema is REQUIRED when using dbtable with Neo4j JDBC
# Without it, Spark schema inference returns NullType() for all columns,
# causing "No column has been read prior to this call" error when reading data.
# Adjust column names and types to match your actual Neo4j node properties.
# NOTE: Use backticks around column names with special characters (like $)
AIRCRAFT_SCHEMA = "`v$id` STRING, aircraft_id STRING, tail_number STRING, icao24 STRING, model STRING, operator STRING, manufacturer STRING"

print(f"\nLabel: {TEST_LABEL}")
print(f"Custom Schema: {AIRCRAFT_SCHEMA}")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("dbtable", TEST_LABEL) \
        .option("customSchema", AIRCRAFT_SCHEMA) \
        .load()
    
    # Force execution and get count
    results = df.take(5)
    total_time = (time.time() - start_time) * 1000
    row_count = len(results)
    
    print("\n" + "=" * 60)
    print(">>> DIRECT JDBC CONNECTION WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] Direct JDBC dbtable '{TEST_LABEL}' read successfully in {total_time:.1f}ms")
    
    print(f"\nDataFrame Schema:")
    for field in df.schema.fields:
        print(f"  - {field.name}: {field.dataType}")
    
    print(f"\nSample Data ({row_count} rows shown):")
    df.show(5, truncate=False)
    
    print(f"Connection Details:")
    print(f"  - Driver: org.neo4j.jdbc.Neo4jDriver")
    print(f"  - URL: {NEO4J_JDBC_URL_SQL}")
    print(f"  - Label/Table: {TEST_LABEL}")
    print(f"  - SQL Translation: Enabled")
    print(f"  - Execution Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: Direct JDBC Connection WORKING")
    print("        Neo4j JDBC Driver loaded and functional")
    print("        SQL-to-Cypher translation enabled")
    print("-" * 60)
    print("\nStatus: PASS")
    
except Exception as e:
    print(f"\n[FAIL] Direct JDBC dbtable failed: {e}")
    print("\nStatus: FAIL")
    print("\nTroubleshooting:")
    print("  1. Ensure neo4j-jdbc-full-bundle JAR is installed as cluster library")
    print("  2. Verify the label exists in Neo4j")
    print("  3. Check customSchema column names match your Neo4j node properties")

In [ ]:
# Direct JDBC - SQL Translation (SQL automatically converted to Cypher)
import time

print("=" * 60)
print("TEST: Direct JDBC - SQL Translation")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Neo4j JDBC translate simple SQL to Cypher?")
print("Query: SELECT 1 AS value")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("query", "SELECT 1 AS value") \
        .option("customSchema", "value INT") \
        .load()
    
    # Force execution
    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    value = results[0]['value'] if results else None
    
    print("\n" + "=" * 60)
    print(">>> SQL TRANSLATION WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] SQL query translated and executed in {total_time:.1f}ms")
    
    print(f"\nQuery Results:")
    df.show(truncate=False)
    
    print(f"Translation Details:")
    print(f"  - Input SQL: SELECT 1 AS value")
    print(f"  - Output Cypher: RETURN 1 AS value (translated automatically)")
    print(f"  - Result Value: {value}")
    print(f"  - Execution Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: SQL-to-Cypher Translation WORKING")
    print("        Neo4j JDBC Driver successfully translates SQL to Cypher")
    print("-" * 60)
    print("\nStatus: PASS")
    
except Exception as e:
    print(f"\n[FAIL] Direct JDBC with SQL translation failed: {e}")
    print("\nStatus: FAIL")

In [ ]:
# Direct JDBC - SQL Aggregate Query (COUNT)
import time

print("=" * 60)
print("TEST: Direct JDBC - SQL Aggregate (COUNT)")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Neo4j JDBC translate SQL aggregates to Cypher?")
print("SQL Query: SELECT COUNT(*) AS flight_count FROM Flight")
print("Expected Cypher: MATCH (n:Flight) RETURN count(n) AS flight_count")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("query", "SELECT COUNT(*) AS flight_count FROM Flight") \
        .option("customSchema", "flight_count LONG") \
        .load()

    # Force execution
    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    count = results[0]['flight_count'] if results else 0

    print("\n" + "=" * 60)
    print(">>> SQL AGGREGATE TRANSLATION WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] SQL aggregate query executed in {total_time:.1f}ms")
    
    print(f"\nQuery Results:")
    df.show(truncate=False)
    
    print(f"Translation Details:")
    print(f"  - Input SQL: SELECT COUNT(*) AS flight_count FROM Flight")
    print(f"  - Translated to Cypher: MATCH (n:Flight) RETURN count(n)")
    print(f"  - Flight Count: {count:,}")
    print(f"  - Execution Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: SQL Aggregate Translation WORKING")
    print("        COUNT(*) on Neo4j label successfully translated to Cypher")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Direct JDBC aggregate query failed: {e}")
    print("\nStatus: FAIL")
    print("\nTroubleshooting:")
    print("  - Ensure 'Flight' label exists in your Neo4j database")
    print("  - Or change the query to use a label that exists in your graph")

In [ ]:
# Direct JDBC - SQL JOIN Translation (NATURAL JOIN -> Cypher relationship)
import time

print("=" * 60)
print("TEST: Direct JDBC - SQL JOIN Translation")
print("=" * 60)
print(f"\nJDBC URL: {NEO4J_JDBC_URL_SQL}")
print("Testing: Can Neo4j JDBC translate SQL JOINs to Cypher relationships?")
print("\nSQL Query:")
print("  SELECT COUNT(*) AS cnt FROM Flight f")
print("  NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a")
print("\nExpected Cypher:")
print("  MATCH (f:Flight)-[:DEPARTS_FROM]->(a:Airport) RETURN count(*)")
print("\nRef: https://neo4j.com/docs/jdbc-manual/current/sql2cypher/")

try:
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("url", NEO4J_JDBC_URL_SQL) \
        .option("driver", "org.neo4j.jdbc.Neo4jDriver") \
        .option("user", NEO4J_USER) \
        .option("password", NEO4J_PASSWORD) \
        .option("query", """SELECT COUNT(*) AS cnt
                           FROM Flight f
                           NATURAL JOIN DEPARTS_FROM r
                           NATURAL JOIN Airport a""") \
        .option("customSchema", "cnt LONG") \
        .load()

    # Force execution
    results = df.collect()
    total_time = (time.time() - start_time) * 1000
    count = results[0]['cnt'] if results else 0

    print("\n" + "=" * 60)
    print(">>> SQL JOIN TRANSLATION WORKING <<<")
    print("=" * 60)
    print(f"\n[PASS] SQL JOIN query translated and executed in {total_time:.1f}ms")
    
    print(f"\nQuery Results:")
    df.show(truncate=False)
    
    print(f"Translation Details:")
    print(f"  - SQL Tables: Flight, DEPARTS_FROM, Airport")
    print(f"  - Cypher Pattern: (f:Flight)-[:DEPARTS_FROM]->(a:Airport)")
    print(f"  - Matching Relationships: {count:,}")
    print(f"  - Execution Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: SQL JOIN to Cypher Relationship Translation WORKING")
    print("        NATURAL JOIN successfully mapped to graph traversal")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Direct JDBC JOIN translation failed: {e}")
    print("\nStatus: FAIL")
    print("\nTroubleshooting:")
    print("  - Requires Flight-[:DEPARTS_FROM]->Airport pattern in Neo4j")
    print("  - Adjust labels/relationship types to match your graph model")